# Phase 4: Full Prescription Inference Demo

This notebook runs the complete inference pipeline from full prescription images to OCR and lexicon-matched medication predictions.

Input: full prescription image(s).

Output: processed pages, region crops, line crops, word crops, OCR predictions, medicine lexicon matching, dosage/frequency extraction, overview images, CSV, and JSON.

## Step 1: Mount Drive, Pull Latest Code, and Enter Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
elif IN_KAGGLE:
    REPO_DIR = Path('/kaggle/working/project')
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print('Repository:', Path.cwd())
print('IN_COLAB:', IN_COLAB, 'IN_KAGGLE:', IN_KAGGLE)
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())

## Step 2: Install Dependencies

Use a GPU runtime for TrOCR inference if available. CPU works for a few words but will be slow.

In [ ]:
!python3 -m pip install -q -r pipeline/requirements.txt
!python3 -m pip install -q -r pipeline/requirements-layout.txt
!python3 -m pip install -q -r pipeline/requirements-ocr.txt

## Step 3: Configure Input Images and Models

Put full prescription images in `data/full_inference_input/`, or change `INPUT_PATH` to one image file.

For final results, set `TROCR_MODEL` to your fine-tuned `best_model` folder. If no fine-tuned model is found, the notebook falls back to base TrOCR for testing only.

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Upload masked full prescription images here in Colab.
INPUT_PATH = Path('data/full_inference_input_masked')
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path('data/full_pipeline_inference_demo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional region detector. If missing, pipeline uses a heuristic region proposal.
REGION_MODEL = Path('models/region_yolo_best.pt')

# Your downloaded Kaggle best-model ZIP stored in Drive/repo.
TROCR_ZIP = Path('data/trocr model/phase4_trocr_best_model.zip')
TROCR_UNZIPPED_DIR = Path('data/trocr model/best_model')

# If the ZIP exists, unzip it once into a folder that transformers can load.
if TROCR_ZIP.exists():
    TROCR_UNZIPPED_DIR.mkdir(parents=True, exist_ok=True)
    required_model_file = TROCR_UNZIPPED_DIR / 'config.json'
    if not required_model_file.exists():
        print('Extracting TrOCR model ZIP:', TROCR_ZIP)
        with zipfile.ZipFile(TROCR_ZIP, 'r') as zf:
            zf.extractall(TROCR_UNZIPPED_DIR)
    else:
        print('TrOCR ZIP already extracted:', TROCR_UNZIPPED_DIR)
else:
    print('TrOCR ZIP not found at:', TROCR_ZIP)

# Fine-tuned TrOCR model candidates.
MODEL_CANDIDATES = [
    TROCR_UNZIPPED_DIR,
    Path('/content/drive/MyDrive/phase4_project/trocr_work/best_model'),
    Path('/kaggle/working/trocr_work/best_model'),
    Path('trocr_work/best_model'),
    Path('models/trocr_best_model'),
]
existing_models = [p for p in MODEL_CANDIDATES if (p / 'config.json').exists()]
TROCR_MODEL = str(existing_models[0]) if existing_models else 'microsoft/trocr-base-handwritten'

LEXICON_PATH = Path('pipeline/config/drug_lexicon.txt')

# Keep this lower for quick demos; use 2200 for final high-quality runs.
MAX_SIDE = 1600
NUM_BEAMS = 1
MAX_TARGET_LEN = 48

print('Input path:', INPUT_PATH.resolve())
print('Input image count:', len([p for p in INPUT_PATH.glob('*') if p.is_file()]))
print('Output dir:', OUTPUT_DIR.resolve())
print('Region model:', REGION_MODEL.resolve(), 'exists=', REGION_MODEL.exists())
print('TrOCR ZIP:', TROCR_ZIP.resolve(), 'exists=', TROCR_ZIP.exists())
print('TrOCR model:', TROCR_MODEL)
print('Lexicon:', LEXICON_PATH.resolve(), 'exists=', LEXICON_PATH.exists())

if TROCR_MODEL == 'microsoft/trocr-base-handwritten':
    print('WARNING: fine-tuned TrOCR model not found. Base TrOCR is only for testing, not final results.')


## Step 4: Upload Full Prescription Images, If Needed

Run this cell in Colab if `data/full_inference_input/` is empty.

In [ ]:
# Upload masked full prescription image(s) for inference.
# This cell always offers upload in Colab. Re-run it when you want to add more images.

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

CLEAR_OLD_INPUTS = True  # set False if you want to keep previous uploaded images
if CLEAR_OLD_INPUTS:
    for p in INPUT_PATH.glob('*'):
        if p.is_file():
            p.unlink()

if IN_COLAB:
    from google.colab import files
    print('Upload masked full prescription image(s).')
    uploaded = files.upload()
    for name, content in uploaded.items():
        out = INPUT_PATH / name
        out.write_bytes(content)
    print('Uploaded files:', list(uploaded.keys()))
else:
    print('Not running in Colab. Put images manually in:', INPUT_PATH)

image_files = [p for p in sorted(INPUT_PATH.glob('*')) if p.suffix.lower() in VALID_EXTS]
print('Images ready for inference:', len(image_files))
for p in image_files:
    print('-', p)


## Step 5: Preview Input Images

In [ ]:
from IPython.display import display, Image as IPImage

image_files = [p for p in sorted(INPUT_PATH.glob('*')) if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff']]
if not image_files:
    raise FileNotFoundError(f'No input images found in {INPUT_PATH}. Upload images or update INPUT_PATH.')

for p in image_files[:5]:
    print(p)
    display(IPImage(filename=str(p), width=700))

## Step 6: Run Full Pipeline Inference

This runs:

full image -> preprocessing -> region crop -> line segmentation -> word segmentation -> TrOCR OCR -> lexicon/dosage/frequency validation.

In [ ]:
import subprocess

if TROCR_MODEL == 'microsoft/trocr-base-handwritten':
    raise RuntimeError('Fine-tuned TrOCR model not found. Check TROCR_ZIP path or unzip model folder before final inference.')

image_files = [p for p in sorted(INPUT_PATH.glob('*')) if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff']]
if not image_files:
    raise FileNotFoundError(f'No input images found in {INPUT_PATH}. Run the upload cell first.')

cmd = [
    'python3', 'pipeline/scripts/run_end_to_end.py',
    '--input', str(INPUT_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--ocr-backend', 'trocr',
    '--ocr-unit', 'word',
    '--trocr-model', str(TROCR_MODEL),
    '--max-target-len', str(MAX_TARGET_LEN),
    '--num-beams', str(NUM_BEAMS),
    '--line-padding', '6',
    '--max-side', str(MAX_SIDE),
    '--lexicon', str(LEXICON_PATH),
]

if REGION_MODEL.exists():
    cmd += ['--yolo-model', str(REGION_MODEL), '--target-class', '0']
else:
    print('Region model missing. Using heuristic region proposal fallback.')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Step 7: Load Output Tables

In [ ]:
import pandas as pd

page_manifest = OUTPUT_DIR / 'page_manifest.csv'
region_manifest = OUTPUT_DIR / 'region_manifest.csv'
line_manifest = OUTPUT_DIR / 'line_manifest.csv'
word_manifest = OUTPUT_DIR / 'word_manifest.csv'
review_csv = OUTPUT_DIR / 'segmentation_review.csv'
pred_csv = OUTPUT_DIR / 'predictions.csv'
filtered_csv = OUTPUT_DIR / 'medicine_dosage_predictions.csv'

pages = pd.read_csv(page_manifest) if page_manifest.exists() else pd.DataFrame()
regions = pd.read_csv(region_manifest) if region_manifest.exists() else pd.DataFrame()
lines = pd.read_csv(line_manifest) if line_manifest.exists() else pd.DataFrame()
words = pd.read_csv(word_manifest) if word_manifest.exists() else pd.DataFrame()
review = pd.read_csv(review_csv) if review_csv.exists() else pd.DataFrame()
pred = pd.read_csv(pred_csv) if pred_csv.exists() else pd.DataFrame()
medicine_pred = pd.read_csv(filtered_csv) if filtered_csv.exists() else pd.DataFrame()

print('Pages:', len(pages))
print('Regions:', len(regions))
print('Lines:', len(lines))
print('Words:', len(words))
print('All OCR predictions:', len(pred))
print('Medicine/dosage rows kept:', len(medicine_pred))

display(medicine_pred.head(50))


## Step 8: Show Segmentation Overview Images

In [ ]:
overview_dir = OUTPUT_DIR / 'segmentation_overview'
print('Overview folder:', overview_dir)

print('Line overviews')
for p in sorted(overview_dir.glob('*line_overview.png'))[:5]:
    print(p)
    display(IPImage(filename=str(p), width=950))

print('Word overviews')
for p in sorted(overview_dir.glob('*word_overview.png'))[:8]:
    print(p)
    display(IPImage(filename=str(p), width=950))

## Step 9: Show OCR Result Table and Validated Medicines

In [ ]:
if pred.empty:
    raise RuntimeError('No predictions found. Check the inference command output above.')

cols = ['page_id', 'region_id', 'line_id', 'word_id', 'image_path', 'ocr_text', 'medicine_name', 'medicine_match_score', 'matched_candidate', 'dosage', 'frequency', 'route', 'validation_status']

print('Medicine names and dosage/frequency only')
if medicine_pred.empty:
    print('No medicine/dosage rows were kept. Check segmentation/OCR output and lexicon coverage.')
else:
    display(medicine_pred[[c for c in cols if c in medicine_pred.columns]].head(80))

matched = medicine_pred[medicine_pred['validation_status'].astype(str) == 'matched'].copy() if 'validation_status' in medicine_pred.columns else pd.DataFrame()
dose_only = medicine_pred[medicine_pred['validation_status'].astype(str) == 'dose_only'].copy() if 'validation_status' in medicine_pred.columns else pd.DataFrame()
print('Matched medicine rows:', len(matched))
print('Dosage/frequency-only rows:', len(dose_only))
display(matched[[c for c in cols if c in matched.columns]].head(40))


## Step 10: Create Demo OCR Grid Image

In [ ]:
import math
import cv2
import numpy as np

preview_dir = OUTPUT_DIR / 'demo_preview'
preview_dir.mkdir(parents=True, exist_ok=True)

preview_source = medicine_pred if not medicine_pred.empty else pred.head(30)

def make_card(row):
    image_path = Path(str(row['image_path']))
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    target_w = 360
    scale = target_w / max(1, w)
    resized = cv2.resize(img, (target_w, max(36, int(h * scale))), interpolation=cv2.INTER_AREA)
    canvas = np.full((resized.shape[0] + 92, target_w, 3), 255, dtype=np.uint8)
    canvas[:resized.shape[0], :, :] = resized
    line1 = f"OCR: {row.get('ocr_text', '')}"[:38]
    line2 = f"Medicine: {row.get('medicine_name', '')}"[:42]
    line3 = f"Dose: {row.get('dosage', '')} {row.get('frequency', '')}"[:42]
    y0 = resized.shape[0]
    cv2.putText(canvas, line1, (8, y0 + 24), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (20, 20, 20), 1, cv2.LINE_AA)
    cv2.putText(canvas, line2, (8, y0 + 52), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 90, 180), 1, cv2.LINE_AA)
    cv2.putText(canvas, line3, (8, y0 + 78), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 120, 0), 1, cv2.LINE_AA)
    return canvas

cards = []
for _, row in preview_source.head(30).iterrows():
    card = make_card(row)
    if card is not None:
        cards.append(card)

if cards:
    cols = 3
    card_h = max(c.shape[0] for c in cards)
    card_w = cards[0].shape[1]
    rows_n = math.ceil(len(cards) / cols)
    grid = np.full((rows_n * card_h, cols * card_w, 3), 245, dtype=np.uint8)
    for idx, card in enumerate(cards):
        y = (idx // cols) * card_h
        x = (idx % cols) * card_w
        grid[y:y+card.shape[0], x:x+card.shape[1]] = card
    preview_path = preview_dir / 'full_pipeline_medicine_dosage_grid.png'
    cv2.imwrite(str(preview_path), grid)
    print('Saved:', preview_path)
    display(IPImage(filename=str(preview_path), width=1000))
else:
    print('No cards created.')


## Step 11: Final Output Summary

In [ ]:
print('Final output folder:', OUTPUT_DIR)
print('All predictions CSV:', OUTPUT_DIR / 'predictions.csv')
print('Medicine/dosage only CSV:', OUTPUT_DIR / 'medicine_dosage_predictions.csv')
print('Medicine/dosage only JSON:', OUTPUT_DIR / 'medicine_dosage_predictions.json')
print('Predictions JSON:', OUTPUT_DIR / 'predictions.json')
print('Segmentation overview images:', OUTPUT_DIR / 'segmentation_overview')
print('Demo OCR grid:', OUTPUT_DIR / 'demo_preview' / 'full_pipeline_medicine_dosage_grid.png')

if 'validation_status' in pred.columns:
    print('All OCR row status counts')
    display(pred['validation_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))
if 'validation_status' in medicine_pred.columns:
    print('Kept medicine/dosage status counts')
    display(medicine_pred['validation_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))
if not review.empty:
    display(review.head(30))


## Optional: Random TrOCR Test-Crop Inference

Run this to test only the TrOCR model on random cropped word images from the combined dataset test split. This does not use full-page segmentation.


In [ ]:
import random
import json
from tqdm.auto import tqdm
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from IPython.display import display, Image as IPImage

TEST_DATASET_CANDIDATES = [
    Path('data/combined_trocr_word_dataset/Testing'),
    Path('/content/drive/MyDrive/phase4_project/repo/data/combined_trocr_word_dataset/Testing'),
    Path('/kaggle/input/handwritten-prescription/combined_trocr_word_dataset/Testing'),
    Path('/kaggle/input/handwritten-prescription/data/combined_trocr_word_dataset/Testing'),
]
TEST_BASE = next((p for p in TEST_DATASET_CANDIDATES if (p / 'testing_labels.csv').exists()), None)
if TEST_BASE is None:
    raise FileNotFoundError('Could not find combined_trocr_word_dataset/Testing. Update TEST_DATASET_CANDIDATES.')

labels_csv = TEST_BASE / 'testing_labels.csv'
words_dir = TEST_BASE / 'testing_words'
test_labels = pd.read_csv(labels_csv)

sample_n = min(5, len(test_labels))
sample_df = test_labels.sample(sample_n, random_state=42).reset_index(drop=True)

processor = TrOCRProcessor.from_pretrained(str(TROCR_MODEL))
model = VisionEncoderDecoderModel.from_pretrained(str(TROCR_MODEL)).to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()
device = next(model.parameters()).device

def norm_text(x):
    return ' '.join(str(x).strip().split())

def predict_crop(path):
    image = Image.open(path).convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = model.generate(pixel_values, max_length=MAX_TARGET_LEN, num_beams=NUM_BEAMS, early_stopping=NUM_BEAMS > 1)
    return norm_text(processor.batch_decode(ids, skip_special_tokens=True)[0])

rows = []
for _, row in sample_df.iterrows():
    img_name = str(row['IMAGE'])
    img_path = words_dir / img_name
    pred_text = predict_crop(img_path)
    label = norm_text(row['MEDICINE_NAME'])
    rows.append({'IMAGE': img_name, 'label': label, 'prediction': pred_text, 'correct': label == pred_text})
    print(img_name, '| label:', label, '| pred:', pred_text)
    display(IPImage(filename=str(img_path), width=240))

random_crop_pred_df = pd.DataFrame(rows)
display(random_crop_pred_df)


## Optional: TrOCR Test Metrics on Combined Test Split

Runs model inference on the combined cropped-word test split and prints exact match, case-insensitive exact match, CER, and WER. Set `MAX_TEST_SAMPLES = None` for the full test split.


In [ ]:
MAX_TEST_SAMPLES = 200  # set None for full test split

metric_df = test_labels.copy()
if MAX_TEST_SAMPLES is not None:
    metric_df = metric_df.sample(min(MAX_TEST_SAMPLES, len(metric_df)), random_state=123).reset_index(drop=True)

metric_rows = []
for _, row in tqdm(metric_df.iterrows(), total=len(metric_df)):
    img_name = str(row['IMAGE'])
    label = norm_text(row['MEDICINE_NAME'])
    pred_text = predict_crop(words_dir / img_name)
    metric_rows.append({'IMAGE': img_name, 'label': label, 'prediction': pred_text})

metric_out = pd.DataFrame(metric_rows)

def edit_distance(a, b):
    a = list(a)
    b = list(b)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[-1][-1]

strict = (metric_out['label'] == metric_out['prediction']).mean()
case_insensitive = (metric_out['label'].str.lower() == metric_out['prediction'].str.lower()).mean()
char_edits = sum(edit_distance(p, y) for p, y in zip(metric_out['prediction'], metric_out['label']))
char_total = sum(max(1, len(y)) for y in metric_out['label'])
word_edits = sum(edit_distance(p.split(), y.split()) for p, y in zip(metric_out['prediction'], metric_out['label']))
word_total = sum(max(1, len(y.split())) for y in metric_out['label'])
cer = char_edits / char_total
wer = word_edits / word_total

metrics = {
    'samples': len(metric_out),
    'strict_exact_match_percent': round(strict * 100, 2),
    'case_insensitive_exact_match_percent': round(case_insensitive * 100, 2),
    'cer_percent': round(cer * 100, 2),
    'wer_percent': round(wer * 100, 2),
}
print(metrics)
display(metric_out.assign(correct=metric_out['label'] == metric_out['prediction']).head(50))

metrics_path = OUTPUT_DIR / 'trocr_test_metrics.json'
preds_path = OUTPUT_DIR / 'trocr_test_predictions_sample.csv'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
metric_out.to_csv(preds_path, index=False)
print('Saved metrics:', metrics_path)
print('Saved predictions:', preds_path)
